In [1]:
from dotenv import load_dotenv
import os

from typing import TypedDict
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END

from langchain_google_genai import ChatGoogleGenerativeAI

In [2]:
load_dotenv()

api_key = os.getenv("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=api_key,
    temperature=0.7
)

In [3]:
class Evaluation(BaseModel):
    score: int = Field(description="Score out of 10")
    feedback: str
    approved: bool


evaluator_llm = llm.with_structured_output(Evaluation)

In [4]:
class LinkedInState(TypedDict):
    topic: str

    post: str

    score: int
    feedback: str

    human_decision: str


In [5]:
def generate_post(state: LinkedInState):

    topic = state["topic"]

    response = llm.invoke(
        f"""
        Write a high quality LinkedIn post on:

        {topic}

        Requirements:
        - Strong hook
        - Storytelling
        - Valuable insight
        - Call to action
        - 150-250 words
        - Professional tone
        """
    )

    return {
        "post": response.content
    }
    
def evaluate_post(state: LinkedInState):

    result = evaluator_llm.invoke(
        f"""
        Evaluate this LinkedIn post.

        Score it out of 10.

        Check:
        - Hook
        - Engagement
        - Readability
        - Professionalism
        - CTA

        Post:

        {state['post']}
        """
    )

    return {
        "score": result.score,
        "feedback": result.feedback
    }
    
def human_review(state: LinkedInState):

    print("\n" + "=" * 50)
    print("GENERATED POST")
    print("=" * 50)

    print(state["post"])

    print("\nAI SCORE:", state["score"])
    print("AI FEEDBACK:", state["feedback"])

    decision = input(
        "\nApprove or Reject? (approve/reject): "
    )

    return {
        "human_decision": decision.lower()
    }
    
def optimize_post(state: LinkedInState):

    response = llm.invoke(
        f"""
        Improve the following LinkedIn post.

        Current Post:
        {state['post']}

        Feedback:
        {state['feedback']}

        Rewrite the complete post.
        """
    )

    return {
        "post": response.content
    }
    
def route_human(state: LinkedInState):

    if state["human_decision"] == "approve":
        return "approved"

    return "rejected"

In [6]:
graph = StateGraph(LinkedInState)

graph.add_node(
    "generate_post",
    generate_post
)

graph.add_node(
    "evaluate_post",
    evaluate_post
)

graph.add_node(
    "human_review",
    human_review
)

graph.add_node(
    "optimize_post",
    optimize_post
)


graph.add_edge(
    START,
    "generate_post"
)

graph.add_edge(
    "generate_post",
    "evaluate_post"
)

graph.add_edge(
    "evaluate_post",
    "human_review"
)

graph.add_conditional_edges(
    "human_review",
    route_human,
    {
        "approved": END,
        "rejected": "optimize_post"
    }
)

graph.add_edge(
    "optimize_post",
    "evaluate_post"
)


workflow = graph.compile()

In [7]:
result = workflow.invoke(
    {
        "topic": "How learning LangGraph helps AI Engineers"
    }
)


print("\n")
print("=" * 50)
print("FINAL POST")
print("=" * 50)

print(result["post"])


GENERATED POST
---

**Hook:** Ever felt your LLM applications hit a wall when faced with complex, multi-step tasks requiring memory, planning, and self-correction?

**Storytelling & Insight:** We've all been there: building fantastic RAG systems or sequential chains, only to find them stumble when an agent needs to autonomously plan, execute, reflect, and *correct* its own actions over multiple turns. This is where the true power of LangGraph shines for AI Engineers.

LangGraph isn't just another library; it's a paradigm shift. It allows you to define stateful, cyclical graphs for your agents, enabling them to maintain context, make decisions, invoke tools, and even loop back for re-evaluation. Think of it as building a sophisticated brain for your AI, moving beyond simple input-output to truly intelligent, adaptive behavior. This means you can engineer robust, production-grade applications that handle complex workflows, manage intricate multi-turn conversations, and significantly red